## Aim:
How to link the rule-based extracted CI_TYPE-GEO pairs with the respective Ci failure impacts

**Idea**\
Test using prompt engineering by passing table of CI-GEO pairs to GPT-J model.
Steps:
* Load model and apply it always on one chunk of the document to extract CI failure impacts 
* Use prompt engineering to extract time and location of the CI failure (origin) the CI impacts (impact location)
or 
* Pass dataframe of pairs as input to the model
or
* Use few shot prompting with example answers

**Finally:**
* Compaire all approaches of spatial and temporal linking CI failure impacts

In [1]:
import os


# # settings for CUDA and PYTORCH
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # settings for debugging CUDA errors (pinpoint exact line of error)
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

# activate global venv explicitly
os.environ["VIRTUAL_ENV"] = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv"


In [2]:
import torch

print(torch.cuda.is_available())
# print(torch.cuda.device_count())  # should give 2
# print(torch.cuda.get_device_name())
# print(torch.cuda.get_device_properties(0))
# print(torch.cuda.get_device_properties(1))
# print(torch.cuda.get_device_capability())
# print(torch.cuda.get_arch_list())
print(torch.__version__)
print(torch.version.cuda)


## --> must be CUDA 12.6, torch: 2.91, ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']


True
2.9.1+cu126
12.6


In [3]:
import os
import sys
import subprocess
import re
import time
from glob import glob
from pathlib import Path
import gc
from io import StringIO
import json

# from tqdm import tqdm
import numpy as np
import pandas as pd
import geonamescache
import pyarrow as pa
import pyarrow.parquet as pq
import langdetect
from langchain_docling import DoclingLoader
# import spacy
from huggingface_hub import login
from pdfminer.high_level import extract_text
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline
from docling.document_converter import DocumentConverter, FormatOption


# sys.path.append("./")
try:
    from src.settings import settings as s
    import src.document_cleaning as dc
    import src.translation_model as tm
    import src.extraction_model as em
    import src.postprocess as pp
    import src.utils as u
    # from submodules.geollama.src.main import GeoLlama
    # from submodules.geollama.src.model import TopoModel, RAGModel

except:
    sys.path.append("/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src")
    
    from src.settings import settings as s
    import src.document_cleaning as dc
    import src.translation_model as tm
    import src.extraction_model as em
    import src.postprocess as pp
    import src.utils as u
    # from submodules.geollama.src.main import GeoLlama
    # from submodules.geollama.src.model import TopoModel, RAGModel

import copy
from jinja2 import Environment, FileSystemLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DynamicCache,
)





test_mode = True

# login to HF
login(token=os.getenv("HUGGINGFACE_TOKEN")) 
# NOTE raises exception if not env.variable doesnt exist (compared to os.envrion.get and its shortcut os.getenv)


# NOTE. disabled batch size as OOM for CUDA despite chunkwise memory cleaning, nvtop to find best batchsize
BATCH_SIZE = s.BATCH_SIZE  # max for nvidia GPU

# torch.manual_seed(42)

#  automatic linebreaks and multi-line cells.
pd.set_option('display.max_colwidth', 100000)
pd.set_option("display.colheader_justify", "left")

print(os.environ["CUDA_VISIBLE_DEVICES"])

# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()

# check if cuda cna be used
# device = transformers.infer_device()
# print(f"Using device: {device}")


# ## TODO test to prevent CUDA-OOM when reused
# ## Source: https://spacy.io/usage/embeddings-transformers
# from thinc.api import set_gpu_allocator, require_gpu

# # Use the GPU, with memory allocations directed via PyTorch.
# # This prevents out-of-memory errors that would otherwise occur from competing
# # memory pools.
# set_gpu_allocator("pytorch")
## require_gpu(0)



Running on TUB cluster
Running on TUB Cluster
0
0.0


## Set paths and vars

In [4]:
# set wd to project root
# os.chdir("/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval")

## set path variables
DOCS_DIR = Path(s.PATH_DATA / "text_sources/")
PARSED_TEXT_DIR = Path(s.PATH_DATA / "parsed_documents/")
NER_PATTERNS_FILEPATH = Path(s.NER_PATTERNS_FILEPATH)
LLM_OUTPUTS_DIR = Path(s.PATH_DATA / "llm_outputs/")
geollama_OUTPUTS_DIR = Path(s.PATH_DATA / "geollama_outputs/")

os.makedirs(PARSED_TEXT_DIR, exist_ok=True)
os.makedirs(s.PATH_LLM_DATA, exist_ok=True)
os.makedirs(geollama_OUTPUTS_DIR, exist_ok=True)


# CI GEO pairs
CI_GEO_FILEPATH = Path( s.PATH_DATA / s.CI_GEO_PAIRS_FILENAME)

## store LLM 1 response and prompt
OUTPUT_LLM1_FILEPATH =  Path(s.PATH_LLM_DATA / s.LLM_DATA_FILENAME)
OUTPUT_geollama_FILEPATH =  Path(s.PATH_LLM_DATA / "geollama_results.csv")




### Set test mode

In [5]:

docs_list_sample = [
        # Path(PARSED_TEXT_DIR, "AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLUVIAS INTENSAS_cleaned.md"),
        # Path(PARSED_TEXT_DIR, "Karakatsani 2023 - Greece economy briefing The economic impact of the recent devastating floods in Greece_cleaned.md"),
        # Path(PARSED_TEXT_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods_cleaned.md"),
        # Path(PARSED_TEXT_DIR, "Containerlift 2024 - Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport_Lifting_Shipping_cleaned.md"), 
        #     Path(PARSED_TEXT_DIR, "ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga_cleaned.md"),

        Path(PARSED_TEXT_DIR, "Koks 2022 - Brief communication_cleaned.md"),
        # Path(PARSED_TEXT_DIR, "European Investment Bank 2025 - Spain_ EIB lends €50 million to Iberdrola to rebuild and climate-proof flood-hit power infrastructure in Valencia_cleaned.md"),
        # Path(PARSED_TEXT_DIR, "Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News_cleaned.md"),     
        #     Path(PARSED_TEXT_DIR, "Wildhagen 2013 - Hochwasser_ Wie die Flut Unternehmen lahmlegt_cleaned.md"),

        # # # not Deidda et al, IPCC, Fekete 2025 as it already contains coarse info about many CI impacts
        #     # Path(PARSED_TEXT_DIR, "AFP 2022 - The_Vibes_Valencia Airport in Madrid briefly shut as lightning hits runway _ World _ The Vibes_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Artemis 2015 - PERILS finalises Storm Desmond UK flood loss estimate at £604m_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Brown 2010 - Economy feels chill as UK grinds to a halt _ The Independent_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Diakakis 2020 - A systematic assessment of the effects of extreme flash floods on transportation infrastructure and circulation: The example of the 2017 Mandra flood_cleaned.md"),
        # Path(PARSED_TEXT_DIR, "EFE 2024 - The DANA storm, live_ The death toll rises to 158_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Eurelectric 2006 - Impacts of Severe Storms on Electric Grids_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Euronews 2024 - Spain floods_ Death toll rises to 205 as nation braces for more rain _cleaned.md"),
        # Path(PARSED_TEXT_DIR, "Ferlita 2023 - Incendi in Sicilia, ecco cosa accade_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Fink 2004 - The 2003 European summer heatwaves and drought - synoptic diagnosis and impacts_cleaned.md"),
        # Path(PARSED_TEXT_DIR, "Gilbody Dickerson 2024 - Spain floods_ At least 95 people killed including British man near Malaga _ World News _ Sky News_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Kadir 2014 - The Impact of Natural Disasters on Critical Infrastructures - A Domino Effect-based Study_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Kaur 2025 - Authorities suspect arson in 17 wildfires across Dalmatian coast, Croatia - The Watchers_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Keller 2014 - Mapping Natural Hazard Impacts on Road Infrastructure—The Extreme Precipitation in Baden-Württemberg_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Kettle 2020 - Storm Xaver over Europe in December 2013 Overview of energy impacts and North Sea events_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Koks 2019 - Understanding Business Disruption and Economic Losses Due to Electricity Failures and Flooding_cleaned.md"),
        #     # Path(PARSED_TEXT_DIR, "Korzilius 2021 Nach der Flut_cleaned.md"),

        # # Path(PARSED_TEXT_DIR, "Khazai 2013 - Juni-Hochwasser 2013 in Mitteleuropa - Fokus Deutschland Bericht 2 Auswirkungen und Bewältigung_cleaned.md"),
            
        #     # not part of valid set:
        #     # Path(PARSED_TEXT_DIR, "Krausmann 2014 - STREST report on lessons learned from recent catastrophic events_cleaned.md"), # > 1800 entries LLMv3.0 incl. hallucinations
]


## Test mode
if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.md")))

print(f"Using {len(search_path)} documents for processing.")

Test mode is ON. Using only a small sample of documents for testing.
Using 1 documents for processing.


###  Load spaCy language model

In [6]:
# ## DOWNLOAD SPACY MODEL and setup PIPELINE just once
# # !uv run python3 -m spacy download en_core_web_trf
# # nlp = spacy.load("en_core_web_trf")

# # # Initialise spaCy pipeline 
# #nlp.add_pipe("merge_entities")
# # nlp.add_pipe("merge_noun_chunks")

# # nlp.to_disk("./spacy_model_pipeline")


# ## RELOAD spacy pipeline
# nlp = spacy.load("./spacy_model_pipeline")

# # add CI_TYPE patterns to spacy nlp model pipeline
# config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
# ## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
# ## NOTE EntityRuler is hidden inside .add_pipe()
# try:
#     ruler = nlp.add_pipe("span_ruler", config=config)
#     ruler.from_disk(s.NER_PATTERNS_FILEPATH)
# except ValueError:
#     print("SpanRuler already exists in pipeline.")
#     ruler = nlp.get_pipe("span_ruler")
#     ruler.from_disk(s.NER_PATTERNS_FILEPATH)


## Document cleaning


In [7]:
# # Docling pipeline configs

# accelerator_options = AcceleratorOptions(
#     num_threads=4, device=AcceleratorDevice.AUTO
# )  # use GPU + multi-threading
# pipeline_options = PdfPipelineOptions()
# pipeline_options.do_ocr = True
# pipeline_options.do_table_structure = (
#     True  # identify tables as such just not to have them in the TextItems later
# )
# pipeline_options.accelerator_options = accelerator_options
# pipeline_options.force_backend_text = True




# # setup converter for PDF and markdown
# converter = DocumentConverter(
#     allowed_formats=[InputFormat.PDF, InputFormat.MD],
#     format_options={
#         InputFormat.PDF: FormatOption(
#             pipeline_cls=StandardPdfPipeline,
#             pipeline_options=pipeline_options,
#             backend=PyPdfiumDocumentBackend,
#         ),
#     },
# )


In [8]:
# print( "Number of documents to process:", len(os.listdir(DOCS_DIR)) )
# start_time = time.time()


# # convert the different layouts of the pdf files into unified markdown format incl. sub/section titles, tables, caption text etc
# for pdf_filename in os.listdir(DOCS_DIR):

#     if pdf_filename.endswith(".pdf"):

#         md_filename = f"{Path(pdf_filename).stem}.md"

#         pdf_filepath = os.path.join(DOCS_DIR, Path(pdf_filename))
#         md_filepath = os.path.join(PARSED_TEXT_DIR, Path(md_filename))
#         cleaned_md_filepath = md_filepath.replace(".md", "_cleaned.md")

#         if os.path.exists(cleaned_md_filepath):
#             print(f"Cleaned markdown file already exists: '{cleaned_md_filepath}'")
#             continue

#         print(f"\nFetching: {pdf_filename}")

#         print("Remove reference section")
#         pdf_text = extract_text(pdf_filepath)
#         pdf_text_no_refs = dc.remove_references(pdf_text)

#         print("Removing URLs") # LangExtract tries to open these URLs when they occur in the document text
#         pdf_text_no_refs_urls = re.sub(r"http\S+", "", pdf_text_no_refs) 

#         # FIXME remove workaround of saving pdf as markdown and reading it again as Docling.Document
#         with open(md_filepath, "w", encoding="utf-8") as f:
#             f.write(pdf_text_no_refs_urls)
#         print("Converting Markdown to text...")
#         # FIXME with DocLoader
#         #loader = DoclingLoader(md_filepath)
#         # md_text = loader.load()
#         md_text = converter.convert(md_filepath)

#         print("Removing headers and footers\n")
#         md_text_cleaned = dc.remove_headers_footers(md_text)

#         ## Translation
#         src_language_doc = langdetect.detect(pdf_filename.replace(".pdf", "").lower())  # lower case improves language detection

#         if src_language_doc != "en":
#             supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
#             if src_language_doc not in supported_languages:
#                 print(f"Unsupported source language: {src_language_doc}. Continue with extraction on original text")
#                 continue 
            
#             print(f"Translating {src_language_doc} --> en")
#             md_text_cleaned.document.texts = tm.translate_2_english(src_language_doc, md_text_cleaned.document.texts)        


#         print(f"Saving parsed and cleaned document as markdown to: {cleaned_md_filepath}")
#         md_text_cleaned.document.save_as_markdown(cleaned_md_filepath)


# end_time = time.time() - start_time
# print(f"Parsing and cleaning done. Time elapsed: {end_time:.2f} seconds.")


# # visual check of removed items
# # TODO make as document_cleaning function: print removed items with largest number of chars first
# # ## NOTE. high number of chars == more potentially actual text body

# # text_items_removed = sorted(text_items_to_drop_visualization, key=lambda x: -x[0])
# # for i in text_items_removed[:50]:
# #     print(i) # -->  also subsection titles were removed partly





## LLama

###  Prompt engineering

In [9]:
question_1 = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, and the type of damage."

### Init Llama application function 
* extract CI-GEO pairs (ie. most likely geolocation of each CI_TYPE )
* pass NER table to prompt for evaluating and improving LLM response
* Test approach by applying it on three cleaned documents


### Load GeoLlama

### Apply Llama on chunks

In [10]:

gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)


0.0


# DecoderModelCaching

In [11]:
# ## Make sure that still both GPUS are visible
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
print(os.environ["CUDA_VISIBLE_DEVICES"])
# # !nvidia-smi


0


In [ ]:




def load_prompt_template(
    template_path: str = "./prompt_templates",
    template_filename: str = None,
):
    env = Environment(loader=FileSystemLoader(template_path))
    template = env.get_template(template_filename)

    return template



class DecoderModelCaching:

    def __init__(self, model_name: str, static_prompt: load_prompt_template):
        
        try: 
            login(token=os.getenv("HUGGINGFACE_TOKEN"))   # notebook_login
        except:
            login(token=os.environ.get("HUGGINGFACE_TOKEN"))  # former HF_TOKEN

        base_dir =  s.HF_HOME_DIR   # use default dir in .cache/
        model_dir = base_dir # / f"models--{model_name.replace("/", "--")}"  # is already .._mirror/hub/
        
        print(model_dir)

        # quantization config
        # Load model with 4-bit quantization if applicable (use 4-bit integer instead of 32b floats) --> reduce the required VRAM for model application
        # see, https://huggingface.co/docs/transformers/quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )

        self.model, self.tokenizer, self.prompt_cache = self.initialize_model(
            model_name, model_dir, bnb_config,
            static_prompt=static_prompt,
        )
        

    def initialize_model(
        self, model_name: str, model_dir: str = None, 
        bnb_config=None, 
        static_prompt=None, 
        max_new_tokens: int = 4096
    ):
            
        # use flash-attn when GPU type supports it (e.g., A100, not support:tesla P100)
        if u.supports_flash_attention(0):  # check only for first GPU
            print("Using flash attention")
            flash_attn_config = "flash_attention_2"
        else:
            print("Flash attention not supported for this GPU")
            flash_attn_config = None

        # Model and Tokenizer initialization
        if not os.path.exists(model_dir):
            print("Model directory not found. Downloading model...")
            os.makedirs(model_dir, exist_ok=True)

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype="auto", # None ,# test for CU12.6, torch.29.1 #"auto",
                # max_seq_length=2048,
                attn_implementation=flash_attn_config,
                quantization_config=bnb_config,
                # max_memory={0: "2GB", 1: "10GB"},  # distribute memory across GPUs
            )
            model.save_pretrained(model_dir)
            
            tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
            tokenizer.save_pretrained(model_dir)

            print("Downloaded model and tokenizer")

        else:
            print(f"Using locally saved model from {model_dir}")

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                cache_dir=model_dir,
                local_files_only=True,  # tp_plan="auto" # set tensor parallel model (ie. splits model on multiple GPU)
                # max_seq_length=2048,
                dtype="auto", # None ,# test for CU12.6, torch.29.1 #"auto",
                attn_implementation=flash_attn_config,
                quantization_config=bnb_config,
                # tp_plan="auto",  # automatically use a tensor parallelism plan based on predefined configuration of the model (i.e. partition model on both GPUs)
            )
            tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                use_fast=True,
                cache_dir=model_dir,  # use fast Rust-based tokenizer, when possible
            )

        ## reduce memory usage
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        model.use_checkpointing = True
        
        torch.cuda.empty_cache()
        torch.no_grad()

        ## Caching
        # set iterative generation to avoid recomputing entire prompt
        # past_key_values = DynamicCache(config=model.config)
        print("Using iterative caching of prompt key values to avoid recomputing entire prompt for each generation step.")
        print("Not using offloading currently") # to CPU for prompt cache to reduce GPU memory usage")
        prompt_cache =  DynamicCache(config=model.config) #, offloading=True) 
        static_prompt = static_prompt.render()
        inputs_initial_prompt = tokenizer(static_prompt, return_tensors="pt").to(model.device.type)
        # This is the common prompt cached, we need to run forward without grad to be able to copy
        with torch.no_grad():
            prompt_cache = model(**inputs_initial_prompt, past_key_values=prompt_cache).past_key_values

        return model, tokenizer, prompt_cache


    def generate_response(
        self, 
        question: str, context: list, 
        static_prompt: load_prompt_template, 
        dynamic_prompt: load_prompt_template, 
        num_beams: int = 1,
        top_k: int = None,
        top_p: float = None,
        temperature: float = 0.2,
        max_new_tokens: int = 1024
    ):
        static_prompt = static_prompt.render()
        dynamic_prompt = dynamic_prompt.render(
            context=context,  # includes also df_ci_geo info
            question=question,
        )
        
        # sequences = self.pipeline(
        #     rendered_prompt,  # jinja template
        #     max_new_tokens=max_new_tokens, # use default to not truncate the LLM response
        #     do_sample=True, # more creative responses
        #     num_beams=1,  # select token based on probability distribution over entire model’s vocabulary
        #     top_k=top_k,
        #     top_p=top_p,
        #     temperature=temperature,
        #     # num_return_sequences=1,
        #     eos_token_id=self.tokenizer.eos_token_id,
        #     # past_key_values=self.past_key_values,
        #     return_full_text=False,  # allow bullet point answers
        # )
        
        new_inputs = self.tokenizer(static_prompt + dynamic_prompt, return_tensors="pt").to(self.model.device.type)

        past_key_values = copy.deepcopy(self.prompt_cache)  # Needed to copy past KV values

        outputs = self.model.generate(
            **new_inputs, 
            temperature=temperature,
            past_key_values=past_key_values, 
            do_sample=True,  # more creative output - NOTE: avoid duplications, better results than when False (greedy decoding)
            max_new_tokens=max_new_tokens
            )

        # Extracting and returning the generated text
        sequences = self.tokenizer.batch_decode(outputs)[0]
        try:
            sequences_cleaned = sequences.split("ANSWER:")[1]
        except:
            sequences_cleaned = sequences.split("The final answer is:")[1]

        return sequences_cleaned


# Run 

In [13]:

# # clean up before applying CUDA
# gc.collect()
# torch.cuda.empty_cache() 
# print(torch.cuda.memory_reserved() / 1e9)
# torch.no_grad()

# model_name = "meta-llama/Llama-3.1-8B-Instruct"
# decoder_model = DecoderModelCaching(
#     model_name,
#     load_prompt_template(template_filename="short_static_llama3.txt",)
# )

In [14]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)

0.0


In [15]:
## Settings
model_name = "meta-llama/Llama-3.1-8B-Instruct"

time0 = time.time()


# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()

## init LLM pipeline
decoder_model = DecoderModelCaching(
    model_name,
    load_prompt_template(template_filename="short_static_llama3.txt",)
)


## init geollama pipeline
## Make sure that still both GPUS are visible
print(os.environ["CUDA_VISIBLE_DEVICES"])


gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)


# CI-GEO pairs
geolocs_cache = geonamescache.GeonamesCache()
countries = geolocs_cache.get_countries()
ci_geo_countries = [*u.gen_dict_extract(countries, 'name')]



## init outputs
df_responses_all_step1 = pd.DataFrame()
responses_error_list = []


if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.md")))


## Start CI impact extraction
for i, filename in enumerate(search_path):

    src_language_nonengl = None 

    no_documents = len(search_path)
    filepath = Path(filename)
    filename_stem = filepath.stem



    ## load doc
    loader = DoclingLoader(filepath)  # use chunks from Docling.Loader
    doc = loader.load()


    print(f"\n\n ######## -------- Processing document [{i+1}/{no_documents}]: {filepath.name} -------- ######## \n")

    ## extract authors, publication year and title 
    author, year, title = dc.extract_citation_info(filename_stem)
    citation = f"{author} {year}".replace("  ", " ").strip()
    title = title.replace(" - ", "").replace("_cleaned", "").strip()



    # print(f"\n ######## -------- Relationship extraction: CI-GEO pairs -------- ######## \n")

    # ## Create New entity for transport infrastructure and apply it on any doc

    # ## TODO make as pydantic class with fixed attributes
    # df_ci_geo = pd.DataFrame(
    #     columns=[
    #         "citation_id",
    #         "chunk_id",
    #         "ci_entity",
    #         "geo_entity",
    #         "case_type",
    #         "chunk_text",
    #         "token_distance",
    #     ]
    # )

    # print(f"\n ######## -------- Getting geolocation of CI assets -------- ######## \n")
    # for chunk_no, chunk in enumerate(doc):

    #     ## preprocess  TODO move to document cleaning workflow + dc.funcs
    #     chunk.page_content = chunk.page_content.replace("\n", " ")
    #     chunk.page_content = chunk.page_content.replace("- ", "-") # TODO test if ("- ", "") is better


    #     ## get most likely geolocation for each CI entity based on distance between tokens
    #     nlp_chunk = nlp(chunk.page_content)
    #     all_ents = [ent for ent in nlp_chunk.ents]
    #     ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]


    #     # check if chunk contains CI_TYPE entities
    #     if len(ci_type_ents) > 0:

    #         # iterate over all entities within chunk
    #         for ent_idx in range(len(all_ents)):
    #             # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
    #             if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
    #                 ci_idx = ent_idx

    #                 ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
    #                 distance_list = []
    #                 idx_in_chunk = []
    #                 try:
    #                     for ent_idx in range(len(all_ents)):
    #                         # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities
    #                         if all_ents[ent_idx].label_ in ["GPE", "LOC"]:

    #                             ## check that GPE,LOC are not countries (too coarse info CI-GEO pair)
    #                             ## of GPE/LOC is country -> proceed with next GPE/LOC 
    #                             if all_ents[ent_idx].text in ci_geo_countries:
    #                                 continue

    #                             geo_idx = ent_idx
    #                             dist_ent_pair = np.abs(ci_idx - geo_idx)
    #                             distance_list.append(dist_ent_pair)
    #                             idx_in_chunk.append((ent_idx))
    #                             closest_pair_idx = np.argmin(
    #                                 distance_list
    #                             )  # idx of closest GEO entity
    #                             distance_closest_pair = distance_list[closest_pair_idx]

    #                     threshold = 5  # max token distance between CI_TYPE and GEO entity
    #                     if distance_closest_pair > threshold:
    #                         # print(
    #                         #     f""" Token distance is too large between CI_TYPE/FAR and next GEO entity which is of {distance_closest_pair} [token distance] > {threshold} [max. token distance] """
    #                         # )
    #                         continue
    #                     else:
    #                         pass

    #                     ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
    #                     result_dict = {
    #                         "chunk_id": chunk_no,
    #                         "chunk_text": chunk.page_content,
    #                         "ci_entity": all_ents[ci_idx].text,
    #                         "ci_entity_label": all_ents[ci_idx].label_,
    #                         "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
    #                         "geo_entity_label": all_ents[idx_in_chunk[closest_pair_idx]].label_,
    #                         "token_distance": distance_closest_pair,
    #                     }
    #                     df_ci_geo = pd.concat(
    #                         [df_ci_geo, pd.DataFrame([result_dict])], ignore_index=True
    #                     )

    #                 except IndexError:
    #                     continue
    #     else:
    #         # print("\nNo CI_TYPE or FAC entities found in this chunk.")
    #         continue

    
    # ## post-process of DF CI-GEO pairs for each chunk
    # unique_ci_geo_pairs = df_ci_geo.drop_duplicates(
    #     subset=["citation_id", "chunk_id", "ci_entity", "geo_entity","case_type", "chunk_text"])
    # print("number of duplicates to remove:", len(df_ci_geo) - len(unique_ci_geo_pairs))

    # df_ci_geo = df_ci_geo.drop_duplicates(
    #     subset=["citation_id", "chunk_id", "ci_entity", "geo_entity","case_type", "chunk_text"]
    #     )# .reset_index(drop=True, inplace=True)



    print(f"\n  #############  -------- Text-2-Data: {filepath.name} -------- #############  \n")

    df_responses_step1 = pd.DataFrame(
        columns=[
            "citation_id",
            "chunk_id",
            "infrastructure_type",
            "damage",
            "location",
            # "ci_entity",
            # "geo_entity",
            "case_type",
            "chunk_text"
        ]
    )


    
    ## apply decoder on each chunk in document
    ## TODO replace iteration by loading entire document and use recursive chunking from langchain
    for j, chunk in enumerate(doc):
        
        print("\n\nCHUNK NO.", j)
        print("Chunk text: ", chunk.page_content)
        time1 = time.time()


        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        torch.no_grad()
        # print(torch.cuda.memory_reserved() / 1e9)
    

        # ## Input for LLM 1  -1st round
        # if df_ci_geo.loc[df_ci_geo["chunk_id"] == j].empty:
        context = [
            {
                "text": chunk.page_content,
                # "citation": citation,
                # "title": filename_stem,
                # "entity_recognition": None,
            },
        ]
        # else:  # TODO dissolve if else clause by making it in ci_locations: if df_ci_geo.chunk=j, xx, else None
        #     context = [
        #         {
        #             "text": chunk.page_content,
        #             "citation": citation,
        #             "title": filename_stem,
        #             "entity_recognition": df_ci_geo.loc[df_ci_geo['chunk_id'] == j],
        #         },
        #     ]

        # load LLM_1 prompt template
        static_prompt = em.load_prompt_template(template_filename="short_static_llama3.txt")
        dynamic_prompt = em.load_prompt_template(template_filename="short_dynamic_llama3.txt")

        # apply LLM 1
        response = decoder_model.generate_response(
            question=question_1, context=context, # chunk_id=j
            static_prompt = static_prompt,
            dynamic_prompt=dynamic_prompt,
            max_new_tokens = 2048
        )
        # print(type(response ))
        # print(response)

        
        ## postprocess response
        try:
            try:
                df_resp = pp.postprocess_response(response[0])
            except (IndexError, ValueError) as e:
                df_resp = pp.postprocess_response(response)
            
            # save LLM response for each chunk  in dataframe for each document
            df_resp["citation_id"] = citation 
            df_resp["chunk_id"] = j  # add chunk id as identifier
            # df_resp["ci_entity"] = df_ci_geo.loc[df_ci_geo["chunk_id"] == j, "ci_entity"] 
            # df_resp["geo_entity"] = df_ci_geo.loc[df_ci_geo["chunk_id"] == j, "geo_entity"]
            # df_resp["case_type"] = df_ci_geo.loc[df_ci_geo["chunk_id"] == j, "case_type"]
            df_resp["chunk_text"] =  context[0]["text"]  # add (translated) chunk text for tracing back LLM response
        
            df_responses_step1 = pd.concat([df_responses_step1, df_resp], ignore_index=True)


        except (IndexError, ValueError) as e:
            print(f"Cannot add response: {e},\nFaulty response (before postprocessing):", response)
            # faulty response: e.g.  .., "location": "V" on satellite and online on radar"}, { ...}, {}
            responses_error_list.append({
                "citation_id": citation,
                "chunk_id": j,
                "response": response,
                "error": str(e)
            })
            print("Continue with next chunk\n")
            continue



        # load NER patterns for CI types and their subgroups
        ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)
        
        ## group Ci types into subgroups,
        if df_resp.infrastructure_type.isna().all():
            print("No infrastructure types extracted for this chunk, skip grouping into subgroups. Go to next chunk")
            continue

        df_resp = pp.group_ci_types(df_resp, "infrastructure_type", "infrastructure_group", ci_patterns)
        ## keep only records which are actually about CI (e.g., not theatre, stadion ..)
        df_resp.dropna(subset=["infrastructure_group"], inplace=True)
    

        # clean up after each chunk
        gc.collect()
        torch.cuda.empty_cache()  # mainly needed after training, small effect when LLM applied only for inference
        torch.no_grad()

        print(f"\nProcessing time for chunk {j}: {np.round(time.time() - time1, 1)} seconds\n")

        if j >= 14:
            break



        ## Input for LLM - 2nd round (geollama)
        # print(f"\nCHECKING input.text, df_resp, geollama resp \n{chunk.page_content}\n{df_resp},\n{locations_per_chunk}")

        context = [
            {
                "text": chunk.page_content,
                "citation": citation,
                "title": filename_stem, 
                "previous_response": df_resp,
                # "potential_locations": locations_per_chunk,  # list of 1 or multiple location strings           
                # "coordinates_potential_locations": list(zip(locations_per_chunk, lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)),
            },
        ]
        

        # response = decoder_model.generate_response(
        #     question=question_1, context=context, # chunk_id=j
        #     prompt_template = template_step2,
        #     temperature = 0.2,
        #     max_new_tokens = 2048
        # )
                
        # ## postprocess response
        # try:
        #     try:
        #         df_resp = pp.postprocess_response(response[0])
        #     except (IndexError, ValueError) as e:
        #         df_resp = pp.postprocess_response(response)
        
        #     # save LLM response for each chunk  in dataframe for each document
        #     df_resp["citation_id"] = citation 
        #     df_resp["chunk_id"] = j  # add chunk id as identifier
        #     # df_resp["ci_entity"] = df_ci_geo.loc[df_ci_geo["chunk_id"] == j, "ci_entity"] 
        #     # df_resp["geo_entity"] = df_ci_geo.loc[df_ci_geo["chunk_id"] == j, "geo_entity"]
        #     # df_resp["case_type"] = df_ci_geo.loc[df_ci_geo["chunk_id"] == j, "case_type"]
        #     df_resp["chunk_text"] =  context[0]["text"]  # add (translated) chunk text for tracing back LLM response
        
        #     # collect resps for each doc
        #     # print("df resp 2")
        #     print(df_resp)

        # except (IndexError, ValueError) as e:
        #     try: 
        #         print(f"Cannot add response: {e},\nFaulty response (before postprocessing):", response)
        #         # faulty response: e.g.  .., "location": "V" on satellite and online on radar"}, { ...}, {}
        #         responses_error_list.append({
        #             "citation_id": citation,
        #             "chunk_id": j,
        #             "response": response, #.replace('\n', ''),
        #             "error": str(e)
        #         })
        #     except:
        #         pass


    
    df_responses_all_step1 = pd.concat([df_responses_all_step1, df_responses_step1], ignore_index=True)        

    # clean up after each document
    gc.collect()
    torch.cuda.empty_cache()  # mainly needed after training, small effect when LLM applied only for inference
    torch.no_grad()


print(f"\n\n ######## -------- CI impact extraction took {(time.time() - time0) / 60} minutes -------- ######## \n\n")


0.0
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub
Flash attention not supported for this GPU
Using locally saved model from /beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Using iterative caching of prompt key values to avoid recomputing entire prompt for each generation step.
Not using offloading currently
0
6.109003776
Test mode is ON. Using only a small sample of documents for testing.


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Token indices sequence length is longer than the specified maximum sequence length for this model (584 > 512). Running this sequence through the model will result in indexing errors




 ######## -------- Processing document [1/1]: Koks 2022 - Brief communication_cleaned.md -------- ######## 


  #############  -------- Text-2-Data: Koks 2022 - Brief communication_cleaned.md -------- #############  



CHUNK NO. 0
Chunk text:  Nat. Hazards Earth Syst. Sci., 22, 3831–3838, 2022 https://doi.org/10.5194/nhess-22-3831-2022 © Author(s) 2022. This work is distributed under the Creative Commons Attribution 4.0 License.
Brief communication: Critical infrastructure impacts of the 2021 mid-July western European ﬂood event
Elco E. Koks1,2, Kees C. H. van Ginkel3,1, Margreet J. E. van Marle3, and Anne Lemnitzer4 1Institute for Environmental Studies, Vrije Universiteit Amsterdam, the Netherlands 2Oxford Programme for Sustainable Infrastructure Systems, Environmental Change Institute, University of Oxford, Oxford, United Kingdom 3Deltares, Delft, the Netherlands 4University of California, Irvine, Irvine, California, United States of America
Correspondence: Kees C. H. van Ginkel (

...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.




CHUNK NO. 14
Chunk text:  We found no information regarding direct impact on solid- waste facilities as a result of the ﬂood event. However, there is a large pressure on the solid-waste sector to clean the af- fected areas; 1 month after the event, we observed dozens of large temporary waste ﬁlls and frequent incidences of oil pollution in Rhineland-Palatinate during a ﬁeld visit. In the Ahrweiler district alone, the ﬂood caused as much solid waste as normally would be collected over 30 years. In Belgium, the amount of solid waste is estimated around 160 000 t, stored at several places, such as the abandoned highway track A601. This highway has been used for approx- imately 9 months as a temporary storage for debris (Cou- plez, 2022). In the Netherlands, there have been primarily problems with waste deposits along the river banks, which is mostly the solid waste transported by the river from fur- ther upstream. Thousands of tonnes of tree debris (logs and
Nat. Hazards Earth Syst. Sci

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Cannot add response: Unexpected character found when decoding array value (1),
Faulty response (before postprocessing):  
[
    {
        "infrastructure_type_1": "solid-waste facilities",
        "damage_1": "NAN",
        "location_1": "Rhineland-Palatinate",
    },
    {
        "infrastructure_type_2": "solid-waste facilities",
        "damage_2": "NAN",
        "location_2": "Ahrweiler district",
    },
    {
        "infrastructure_type_3": "solid-waste facilities",
        "damage_3": "NAN",
        "location_3": "Belgium",
    },
    {
        "infrastructure_type_4": "solid-waste facilities",
        "damage_4": "NAN",
        "location_4": "Netherlands",
    },
    {
        "infrastructure_type_5": "highway",
        "damage_5": "NAN",
        "location_5": "abandoned highway track A601",
    },
]  # STEP 1: 
You must first determine if the CONTEXT describes an impact or damage to a critical infrastructure
Carefully read the CONTEXT. Then, decide if the CONTEXT describes an 

KeyboardInterrupt: 

0    True
Name: infrastructure_type, dtype: bool

In [17]:
df_responses_step1.tail(3)

,citation_id,chunk_id,infrastructure_type,damage,location,case_type,chunk_text
128,Koks 2022,13,water supply infrastructure,rebuild,Netherlands,NaN,"industrial usage (Kuhn, 2021). In the heavily destroyed town of Bad Münstereifel (in the state of North Rhine-Westphalia), drinking water supply was re-established within 5 d after the ﬂood event (most frequently through emergency tanks), and about 50 % of the city centre was reconnected to the fresh- water network shortly thereafter however, water had to be boiled before consumption until about 1 month later (Bad Münstereifel, 2021). In Belgium, several towns experienced disruptions in water supply (in particular as a result of pol- lution). Directly after the event, approximately 3400 families had no access to potable water. Within less than a week, this was reduced to around 1650 families (Terzake, 2021). It took, however, 6 months to rebuild the permanent water supply in- frastructure (SWDE, 2022). In the Netherlands, little to no problems have been recorded with regards to water supply."
129,Koks 2022,13,water supply infrastructure,rebuild,Netherlands,NaN,"industrial usage (Kuhn, 2021). In the heavily destroyed town of Bad Münstereifel (in the state of North Rhine-Westphalia), drinking water supply was re-established within 5 d after the ﬂood event (most frequently through emergency tanks), and about 50 % of the city centre was reconnected to the fresh- water network shortly thereafter however, water had to be boiled before consumption until about 1 month later (Bad Münstereifel, 2021). In Belgium, several towns experienced disruptions in water supply (in particular as a result of pol- lution). Directly after the event, approximately 3400 families had no access to potable water. Within less than a week, this was reduced to around 1650 families (Terzake, 2021). It took, however, 6 months to rebuild the permanent water supply in- frastructure (SWDE, 2022). In the Netherlands, little to no problems have been recorded with regards to water supply."
130,Koks 2022,13,water supply infrastructure,rebuild,Netherlands,NaN,"industrial usage (Kuhn, 2021). In the heavily destroyed town of Bad Münstereifel (in the state of North Rhine-Westphalia), drinking water supply was re-established within 5 d after the ﬂood event (most frequently through emergency tanks), and about 50 % of the city centre was reconnected to the fresh- water network shortly thereafter however, water had to be boiled before consumption until about 1 month later (Bad Münstereifel, 2021). In Belgium, several towns experienced disruptions in water supply (in particular as a result of pol- lution). Directly after the event, approximately 3400 families had no access to potable water. Within less than a week, this was reduced to around 1650 families (Terzake, 2021). It took, however, 6 months to rebuild the permanent water supply in- frastructure (SWDE, 2022). In the Netherlands, little to no problems have been recorded with regards to water supply."


In [ ]:
## apply CI grouping as respesones_all_1 not contains cleaned version


# load NER patterns for CI types and their subgroups
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

## group Ci types into subgroups,
df_responses_step1_ci = pp.group_ci_types(df_responses_step1, "infrastructure_type", "infrastructure_group", ci_patterns)
## keep only records which are actually about CI (e.g., not theatre, stadion ..)
df_responses_step1_ci.dropna(subset=["infrastructure_group"], inplace=True)


In [ ]:

## saving
df_responses_step1.to_csv("df_responses_step1_no_ci.csv", index=False)
df_responses_step1_ci.to_csv("df_responses_step1_ci.csv", index=False)

infrastructure_type
railway infrastructure         57
road infrastructure            24
bridge(s)                      10
motorway(s)                     9
water infrastructure            5
water supply infrastructure     3
electricity infrastructure      2
drinking water supply           2
sewage system(s)                1
school(s)                       1
hospital(s)                     1
railways                        1
bridges                         1
hospitals                       1
telecommunication services      1
gas distribution network        1
sewage treatment plant          1
wastewater facilities           1
groundwater                     1
water supply                    1
Name: count, dtype: int64

: 

1. FIX CI hallucis: rerun LLm with only CI-geo pairs (no geollama) and org.workflow and prompt (fix cigeo-dublicates)

* currently-local-fixedGROUP_CI check if old WF (oldWF vs newWF) is better than this one here (newWF) - similar scores in CI recall:0.5, f1:0.14-0.17
<!-- 
# ### NEW WF

<!-- ### OLD WF

# 
(maybe solved via geonamescache) --> LLM-step1 summarizes loc too much (belgium, instead of Orte --> soll idese als einzl dict rausschrieben 

#### --> test OUTP als list od dic mit ci1, loc1, ci2, loc2 ...) falls org LLM-ci-geo zu selben problem führt

currently 2. re-add geollama  (step2)  (+ prompt to recheck CI correct link to loc)

### 3. fix location coords e.g. by selecting "location" from retunred list_of_potential_locs

### 4. Doc cleaning (rm Abstracts, and footers Nat.Haz. ..)




### FIX. check geollama ~ llm_1 interaction

In [ ]:

print(df_responses_step2.infrastructure_group.isna().sum())  # mostly cases which are not CI (theater, stadion..)
print(df_responses_step2.infrastructure_group.value_counts()) # four most common subgroups seems to be correct
# df_pred.infrastructure_group.unique()



In [ ]:
df_responses_step2
# df_responses_all_step2
# df_responses_step2.loc[20:30, ["infrastructure_type", "infrastructure_group", "damage", "location", "chunk_text"]]


In [ ]:

gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)

In [ ]:
# df_ci_geo
j = 3
df_resp["ci_entity"] = df_ci_geo.loc[df_ci_geo["chunk_id"] == j, "ci_entity"]
df_resp.sort_values(by="ci_entity")

In [ ]:
# 12 docs with fixed NERpatterns: 74min

In [ ]:
print(torch.cuda.memory_reserved() / 1e9)


In [ ]:
print("Chunk with erroneous responses:", responses_error_list.__len__())
df_responses_error = pd.DataFrame(responses_error_list)
df_responses_error#.tail(3)


In [ ]:
df_responses_all_step2#.tail(3)

### Saving

In [ ]:
# PATH_LLM_DATA: Path = Path(s.PATH_DATA /"llm_outputs/")
# LLM_DATA_FILENAME: str = "llm_1_updprompt_distanceNER.csv"

# OUTPUT_LLM1_FILEPATH =  Path(PATH_LLM_DATA / LLM_DATA_FILENAME)#.replace(".csv", "_v2.csv"))
# OUTPUT_LLM1_FILEPATH 
df_responses_step1.to_csv("1doc_llama3.csv", index=False)
# df_responses_step1.to_csv("1doc_llama3_ner.csv", index=False)
# df_responses_step2.to_csv("llm1_geollama_step2_partly.csv", index=False)


In [ ]:
safety_df = df_responses_all_step2.copy()



# save LLM 1 output to disk along with prompt text
if not os.path.isfile(OUTPUT_LLM1_FILEPATH):

    print(f"Saving prompt, LLM response and erroneous responses [.txt, .csv] to {OUTPUT_LLM1_FILEPATH} ...")

    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(template_1.render(context=context, question=question_1))
    df_responses_all_step2.to_csv(OUTPUT_LLM1_FILEPATH, index=False)
    df_responses_error.to_csv(OUTPUT_LLM1_FILEPATH.parent / f"errors_{OUTPUT_LLM1_FILEPATH.stem}.csv", index=False)

elif os.path.isfile(OUTPUT_LLM1_FILEPATH) and not os.path.isfile(OUTPUT_LLM1_FILEPATH.parent / f"{OUTPUT_LLM1_FILEPATH.stem}_v2.csv"):

    print(f"Output file {Path(OUTPUT_LLM1_FILEPATH).stem} already exists. Saving as {OUTPUT_LLM1_FILEPATH.stem}_v2 to avoid overwriting ...")

    # If the original files exists but the v2 file doesn't, create the v2 file
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(template_1.render(context=context, question=question_1))
    df_responses_all_step2.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent, f"{OUTPUT_LLM1_FILEPATH.stem}_v2.csv"), index=False)
    df_responses_error.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent / f"errors_{OUTPUT_LLM1_FILEPATH.stem}_v2.csv"), index=False)

else:
    print(f"Output file {Path(OUTPUT_LLM1_FILEPATH).stem} already exists. Skip saving to avoid overwriting ...")






Save as pyarrow incl dtypes as pyarraws-. Saving also dtypes a spyarrow objectes leads to quicker loading and processing

In [ ]:
# pd.options.future.infer_string = True

df_responses_all_pyarrow = df_responses_all_step2.astype( dtype="string[pyarrow]")
df_responses_all_pyarrow = pa.Table.from_pandas(df_responses_all_pyarrow)
pq.write_table(df_responses_all_pyarrow, OUTPUT_LLM1_FILEPATH.with_suffix(".parquet"))  



In [ ]:
response

In [ ]:
# df_ci_geo
j = 3
df_resp["ci_entity"] = df_ci_geo.loc[df_ci_geo["chunk_id"] == j, "ci_entity"]
df_resp.sort_values(by="ci_entity")

In [ ]:
# 12 docs with fixed NERpatterns: 74min

In [ ]:
print(torch.cuda.memory_reserved() / 1e9)


In [ ]:
print("Chunk with erroneous responses:", responses_error_list.__len__())
df_responses_error = pd.DataFrame(responses_error_list)
df_responses_error#.tail(3)


In [ ]:
df_responses_all_step2#.tail(3)